In [1]:
import bisect
import pandas as pd
import json
# a: the original whole set list
# b: the 6060 dev set list
df_a = pd.read_json("/home/jiaruil5/multilingual/multilingual-model-card/src/dictionary_collection/growing_dict/mturk.json").sort_values(by='English')
df_a['source'] = "whole"
df_b = pd.read_csv("/home/jiaruil5/multilingual/multilingual-model-card/src/dictionary_collection/mturk/terminology_from_6060.csv")

# get context of b
context_b = [i.strip() for i in open("/home/jiaruil5/multilingual/multilingual-model-card/src/data_eval_6060/2/acl_6060/dev/text/tagged_terminology/ACL.6060.dev.tagged.en-xx.en.txt", 'r').readlines()]
df_b_upd_lst = []
for idx, row in df_b.iterrows():
    term = row['english']
    context_str_lst = []
    for context in context_b:
        if f"[{term}]" in context:
            context = context.replace("[", "")
            context = context.replace("]", "")
            context = context.replace(term, f"<mark>{term}</mark>")
            context_str_lst.append(context.strip())
    context_str = "<br>".join([f"{i+1}: " + item for i, item in enumerate(context_str_lst[:3])]) + "<br>"
    new_info = {
        "English": term,
        "context": context_str,
        "Arabic": row['arabic'],
        "Chinese": row['chinese'],
        "French": row['french'],
        "Japanese": row['japanese'],
        "Russian": row['russian'],
        "source": "6060"
    }
    df_b_upd_lst.append(new_info)
df_b_upd = pd.DataFrame().from_records(df_b_upd_lst)

In [2]:
a = df_a.to_dict(orient='records')
b = df_b_upd.to_dict(orient='records')

In [3]:
import heapq
a.sort(key=lambda x: x['English'])
b.sort(key=lambda x: x['English'])

# Merge the two sorted lists
merged = list(heapq.merge(a, b, key=lambda x: x['English']))

In [4]:
merged

[{'English': '10-fold cross validation',
  'context': "1: For all the methods, we used <mark>10-fold cross validation</mark> (i.e., each fold we have 556 training and 62 test samples) to tune free parameters, e.g., the kernel form and parameters for GPOR and LapSVM. Note that all the alternative methods stack X and Z together into a whole data matrix and ignore their heterogeneous nature.<br>2: Features associated one-to-one with a vertical (Clarity, ReDDE, the query likelihood given the vertical's query-log and Soft.ReDDE) were normalized across verticals before scaling. Supervised training/testing was done via <mark>10-fold cross validation</mark>. Parameter τ was tuned for each training fold on the same 500 query validation set used for our single feature baselines.<br>",
  'Arabic': 'التحقق المتقاطع بمقدار ١٠ أضعاف',
  'Chinese': '十折交叉验证',
  'French': 'validation croisée 10 fois',
  'Japanese': '10分割交差検証',
  'Russian': '10-кратная перекрестная проверка',
  'source': 'whole'},
 {'En

In [4]:
df = pd.DataFrame().from_records(merged)

In [8]:
df.shape[0]

5027

In [12]:
df[df.duplicated(subset=['English'], keep=False)]

,English,context,Arabic,Chinese,French,Japanese,Russian,source
895,annotation,1: The '<mark>annotation</mark>' prompt contai...,تعليقات,标注,annotation,アノテーション,аннотация,whole
896,annotation,1: This could be done by human <mark>annotatio...,التعليق التوضيحي,注释,annotation,注釈,аннотация,6060
973,augmentation,1: zero mean unit variance Gaussian random var...,تكبير,增强,augmentation,拡張,усиление,whole
974,augmentation,"1: So, to effectively evaluate the effectivene...",زيادة,增强,élargissement,増大,увеличение,6060
1084,bidirectional,1: The placement of an information bottleneck ...,ثنائي الاتجاه,双向的,bidirectionnel,双方向性,двунаправленный,whole
...,...,...,...,...,...,...,...,...
4867,utterance,1: From the finish of the user <mark>utterance...,نُطقٍ,发声,énonciation,発話,высказывание,6060
4906,vector,1: Once we had learned a <mark>vector</mark> w...,المتجه,向量,vecteur,ベクトル,вектор,whole
4907,vector,1: The other option is to predict the affix pr...,متجه,向量,vectoriel,ベクトル,вектор,6060
4947,vocabulary,"1: For the En→De translation task, sentences a...",مفردات,词汇量,vocabulaire,語彙,словарь,whole


In [10]:
df['English'].unique().shape

(4990,)

In [6]:
df.to_csv("/home/jiaruil5/multilingual/multilingual-model-card/src/dictionary_collection/mturk/mturk_with_6060.csv")

In [6]:
# df = df.sample(frac=1).reset_index(drop=True)

In [7]:
# chunk mturk
num_terms_per_hit = 10
# Create empty sub-dataframes
interleaved_sub_dfs = [df.iloc[i::num_terms_per_hit].reset_index(drop=True) for i in range(num_terms_per_hit)]

# Add suffixes to each interleaved sub-dataframe
suffixes = [str(i) for i in range(1, num_terms_per_hit+1)]
interleaved_sub_dfs = [sub_df.add_suffix(suffix) for sub_df, suffix in zip(interleaved_sub_dfs, suffixes)]

# Concatenate the interleaved sub-dataframes by columns
interleaved_concat_df = pd.concat(interleaved_sub_dfs, axis=1)

interleaved_concat_df

,English1,context1,Arabic1,Chinese1,French1,Japanese1,Russian1,source1,English2,context2,...,Russian9,source9,English10,context10,Arabic10,Chinese10,French10,Japanese10,Russian10,source10
0,10-fold cross validation,"1: For all the methods, we used <mark>10-fold ...",التحقق المتقاطع بمقدار ١٠ أضعاف,十折交叉验证,validation croisée 10 fois,10分割交差検証,10-кратная перекрестная проверка,whole,1D convolution,1: In addition to the usual quadratic kernels ...,...,трехмерное компьютерное зрение,whole,3D convolutional network,1: The choice of architecture proves to be imp...,شبكة تلافيفية ثلاثية الأبعاد,三维卷积网络,réseau convolutionnel 3D,3次元畳み込みネットワーク,трёхмерная сверточная сеть,whole
1,3D geometry,1: To that end we define an energy function th...,هندسة ثلاثية الأبعاد,三维几何,géométrie 3D,3次元ジオメトリ,3D геометрия,whole,3D human pose estimation,1: Existing diffusion-based pose estimation ap...,...,3D поза,whole,3D reconstruction,1: Minimal problems arise from geometrical pro...,إعادة الإعمار ثلاثية الأبعاد,三维重建,reconstruction 3D,3D再構築,Трехмерная реконструкция,whole
2,3D scene,1: Since most current scene understanding appr...,المشهد ثلاثي الأبعاد,三维场景,scène 3D,3次元シーン,трёхмерная сцена,whole,3D scene geometry,"1: However, exhaustive sampling of the light f...",...,API,6060,AQA,1: Apart from the three datasets <mark>AQA</ma...,AQA,AQA,AQA,AQA,AQA,6060
3,ARENA,<br>,ARENA,ARENA,ARENA,ARENA,ARENA,6060,Ablation study,1: <mark>Ablation study</mark>: Table 4 shows ...,...,Оптимизация Адама,whole,Adam optimization algorithm,1: This loss function is only used during trai...,خوارزمية التحسين آدم,Adam优化算法,algorithme d'optimisation Adam,Adamの最適化アルゴリズム,Алгоритм оптимизации Адама,whole
4,Adam optimizer,"1: Second, the <mark>Adam optimizer</mark> in ...",محسن آدم,Adam优化器,Optimiseur Adam,Adamオプティマイザ,Оптимизатор Адам,whole,Adapter,"1: In <mark>Adapter</mark>-based IPT, we set t...",...,аркадная обучающая среда,whole,Architectures,<br>,هياكل,架构,architectures,アーキテクチャ,Архитектуры,6060
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,weight vector,"1: where n is the number of training examples,...",متجه الوزن,权重向量,vecteur de poids,重みベクトル,вектор весов,whole,weight-sharing,1: Siamese networks can naturally introduce in...,...,веса,6060,white-box,1: Next we employ them as the trigger to attac...,صندوق أبيض,白盒,boîte blanche,ホワイトボックス,белый ящик,whole
499,white-box attack,1: Defense-GAN was not shown to be effective o...,هجوم الصندوق الأبيض,白盒攻击,attaque en boîte blanche,ホワイトボックス攻撃,атака белого ящика,whole,window size,1: The performance of kb-SRK reaches the peak ...,...,представление слова,whole,word segmentation,1: We present an unsupervised <mark>word segme...,تجزئة الكلمة,词语分割,segmentation des mots,単語分割,сегментация слов,whole
500,word sense disambiguation,1: named entity recognition (Collins and Singe...,تمييز معنى الكلمة,词义消歧,désambiguïsation du sens des mots,単語意味の曖昧さ解消,разрешение неоднозначности слова,whole,word similarity,"1: In this paper, we describe design considera...",...,словарь на уровне слов,whole,word2vec,<br>,word2vec,word2vec,word2vec,単語2vec,word2vec,6060
501,word2vec embedding,1: The use of lexical semantic information in ...,تضمين word2vec,词向量嵌入,représentation word2vec,word2vecエンベディング,word2vec вложения,whole,words,"1: Well, lexical borrowing is basically the in...",...,нулевая межъязыковая настройка,whole,zero-shot generalization,1: We also observe that the EM of ArcaneQA on ...,التعميم الصفري,零样本泛化,généralisation zéro-shot,ゼロショット汎化,обобщение с нулевым выстрелом,whole


In [8]:
interleaved_concat_df.to_csv("mturk_10terms_per_page_with_6060.csv")